In [ ]:
import logging
import warnings

import itertools
import numpy

from library.circuitry import DetectorTag

logging.basicConfig(level=logging.ERROR)
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
from IPython.core.display import Markdown
from IPython.display import display

from library.qubit_array import QubitArray
from library.surface_code.patch import SurfaceCodePatch
from library.circuitry import Circuitry
from library.compact_magic_state_cultivation import MagicStateCultivation
from library.utils.clifft.stratified_mc_reweighted import compute_ler_curves
from library.utils.clifft.plotting import (
    plot_logical_error_rates,
    plot_stratum_data,
)

In [ ]:
TARGET_DISTANCE = 7

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

TELEPORT_ROUNDS = 3
minimum_anchoring = 1 + int(TARGET_DISTANCE == 7)
TARGET_ANCHOR = (minimum_anchoring, minimum_anchoring)

scenarios = {}

In [ ]:
qubits: QubitArray = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=0, exp_rounds=0)

msc.steane.append_noiseless_measurement(circuitry)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Injected"] = circuitry

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=0)

msc.steane.annotate_stabilizers(circuitry, style="DIFFUSE", noiseless=True)
for stabilizer, color in itertools.product("XZ", "RGB"):
    circuitry.annotate_detector(
        f"STN:{stabilizer}{color}", tags=DetectorTag.POSTSELECTED
    )

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Superdense"] = circuitry

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=0)

msc.steane.annotate_stabilizers(circuitry, style="DIFFUSE", noiseless=True)
for stabilizer, color in itertools.product("XZ", "RGB"):
    circuitry.annotate_detector(
        f"STN:{stabilizer}{color}", tags=DetectorTag.POSTSELECTED
    )

msc.steane.append_noiseless_measurement(circuitry, compact=False)
circuitry.attach_observable(0, "STN:NOISELESS:CC")

scenarios["Double-check"] = circuitry

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)
msc.append_teleportation(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=0)

msc.source.annotate_stabilizers(circuitry, prefix="SRC:STB", noiseless=True)
circuitry.annotate_detector(
    "JCT:TPT2:X0", "SRC:STB:X0", tags=DetectorTag.COMPLEMENTARY_GAP
)
for stabilizer in "XZ":
    for location, (_, index) in msc.source.qubits[stabilizer].items():
        tags = DetectorTag.COMPLEMENTARY_GAP
        if msc.critical_on_seam(stabilizer, location):
            tags |= DetectorTag.POSTSELECTED
        circuitry.annotate_detector(
            f"SRC:TPT2:{stabilizer}{index}",
            f"SRC:STB:{stabilizer}{index}",
            tags=tags,
        )
msc.source.append_decode_measurement(
    circuitry,
    prefix="SRC:DCD",
    noiseless=True,
    corrections=(f"JCT:TPT2:{mz}" for mz in ["Z0", "Z1", "Z2"]),
)

circuitry.attach_observable(
    0,
    "SRC:DCD:X",
    *(f"STN:TPT2:X{mx}" for mx in [1, 5, 6]),
)

scenarios["Teleported"] = circuitry

In [ ]:
qubits = SurfaceCodePatch.make_array(TARGET_DISTANCE, (2, 2))
circuitry = Circuitry(qubits)
magic = SurfaceCodePatch(
    qubits, distance=TARGET_DISTANCE, anchor=(minimum_anchoring, minimum_anchoring)
)
msc = MagicStateCultivation(qubits, target=magic)

msc.append_injection(circuitry)
msc.append_superdense_cycle(circuitry, prefix="STN:SDC0")
msc.append_doublecheck(circuitry)
msc.append_teleportation(circuitry)
exp_rounds = msc.append_expansion(circuitry)

msc.annotate_detectors(circuitry, sdc_rounds=1, exp_rounds=exp_rounds)

msc.target.annotate_stabilizers(circuitry, prefix="TGT:STB", noiseless=True)
for stabilizer, index in itertools.product(
    "XZ", range((msc.target.distance**2 - 1) // 2)
):
    circuitry.annotate_detector(
        f"TGT:EXP:R{exp_rounds - 1}:{stabilizer}{index}",
        f"TGT:STB:{stabilizer}{index}",
        tags=DetectorTag.COMPLEMENTARY_GAP,
    )

msc.target.append_decode_measurement(
    circuitry,
    prefix="TGT:DCD",
    noiseless=True,
    corrections=(f"JCT:TPT2:Z{i}" for i in range(3)),
    offset=msc.target.distance - msc.source.distance,
)

circuitry.attach_observable(
    0,
    "TGT:DCD:X",
    *(f"STN:TPT2:X{mx}" for mx in [1, 5, 6]),
)

# scenarios["Expanded"] = circuitry

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    label = f"Point {index} - {scenario}"
    display(Markdown(f"[Open in Crumble ({label})]({circuitry.to_crumble_url()})"))

In [ ]:
strata_count = 16
noise_levels = numpy.logspace(-7, -2, num=11)
ler_curves, cer_distributions = compute_ler_curves(
    scenarios,
    noise_levels=noise_levels,
    strata_count=strata_count,
    shots_per_stratum=1e6,
    batch_count=20,
    max_errors=100,
    gap_threshold=48.0,
)

In [ ]:
plot_logical_error_rates(ler_curves, cer_distributions, label="Point")

In [ ]:
plot_stratum_data(cer_distributions, label="Point", figsize=(14, 14))

In [ ]:
import library.utils.clifft.plotting as plotting

plotting.plot_overhead_curves(ler_curves, label="Point")